In [92]:
import torch
import torch.nn as nn
import torch.nn.functional as F

device = 'cuda' if torch.cuda.is_available() else 'cpu'

print('Using {} device'.format(device))

block_size = 8
batch_size = 4
max_iters = 50000
learning_rate = 3e-4
eval_iters = 5000

Using cpu device


In [93]:
with open('data/dutch_bible.txt', 'r', encoding='utf8') as file:
    text = file.read()
    
chars = sorted(set(text))
print(chars)
vocabulary_size = len(chars)
print(vocabulary_size)

['\n', ' ', '!', '"', '&', "'", '(', ')', '+', ',', '-', '.', '/', '0', '1', '2', '3', '4', '5', '6', '7', '8', '9', ':', ';', '=', '>', '?', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'Y', 'Z', '`', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z', '|', 'ä', 'è', 'é', 'ë', 'ï', 'ò', 'ü']
88


In [94]:
string_to_int = { ch:i for i, ch in enumerate(chars) }
int_to_string = { i:ch for i, ch in enumerate(chars) }

encode = lambda s: [string_to_int[ch] for ch in s]
decode = lambda x: ''.join([int_to_string[i] for i in x])

data = torch.tensor(encode(text), dtype=torch.long)

In [95]:
# Defines the split (80% train, 20% validation)
n = int(0.8 * len(data))
train_data = data[:n]
validation_data = data[n:]

def get_batch(split):
    data = train_data if split == 'train' else validation_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y

x, y = get_batch('train')

In [96]:
class BigramLanguageModel(nn.Module):
    def __init__(self, vocabulary_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocabulary_size, vocabulary_size)
        
    def forward(self, index, targets=None):
        logits = self.token_embedding_table(index)
        
        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)
        
        return logits, loss
    
    def generate(self, index, max_new_tokens):
        for _ in range(max_new_tokens):
            logits, loss = self.forward(index)
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=-1)
            index_next = torch.multinomial(probs, num_samples=1)
            index = torch.cat((index, index_next), dim=1)
            
        return index
    
model = BigramLanguageModel(vocabulary_size)
m = model.to(device)

context = torch.zeros((1, 1), dtype=torch.long, device=device)
print(context)
generated_chars = decode(m.generate(context, max_new_tokens=100)[0].tolist())
print(generated_chars)

tensor([[0]])

B6br6/E!-bc5Jf-lEg-SQ=i(k1hu;ëiKkVe7&rmwDx.jKNa/N?ä5IO/ed(vVésJI|l,0+.F:;aAMe'-" ,rt
c>:kzwü'qüL=hH2


In [97]:
@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

In [98]:
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

for iter in range(max_iters):
    if iter % eval_iters == 0:
        losses = estimate_loss()
        print(f'Iter {iter}, train loss: {losses["train"]}, val loss: {losses["val"]}')
    
    xb, yb = get_batch('train')
    logits, loss = model.forward(xb, yb)
    # Optimises and sets the gradients to zero so more efficient because int instead of float
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

print(loss.item())

Iter 0, train loss: 4.975143909454346, val loss: 4.974095821380615
Iter 5000, train loss: 3.8474245071411133, val loss: 3.841266393661499
Iter 10000, train loss: 3.1458561420440674, val loss: 3.1401965618133545
Iter 15000, train loss: 2.7585558891296387, val loss: 2.7500507831573486
Iter 20000, train loss: 2.5617942810058594, val loss: 2.558387517929077
Iter 25000, train loss: 2.4610633850097656, val loss: 2.4614098072052
Iter 30000, train loss: 2.4057655334472656, val loss: 2.4024569988250732
Iter 35000, train loss: 2.373838424682617, val loss: 2.371659278869629
Iter 40000, train loss: 2.350188732147217, val loss: 2.353365659713745
Iter 45000, train loss: 2.334117889404297, val loss: 2.331064224243164
2.240903854370117


In [108]:
context = torch.zeros((1, 1), dtype=torch.long, device=device)
generated_chars = decode(m.generate(context, max_new_tokens=100)[0].tolist())
print(generated_chars)


6-21: zat me degt Tijng henebst i he zodedopeke optet ge'éPòf, st h dete ndenn g miedist t bremoren 
